In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "proy26";
create widget text catalogwk default "catalog_prod";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS $catalogwk
COMMENT 'Catalogo para la arquitectura medallion';

In [0]:
%sql
DROP SCHEMA IF EXISTS $catalogwk.raw;
DROP SCHEMA IF EXISTS $catalogwk.bronze;
DROP SCHEMA IF EXISTS $catalogwk.silver;
DROP SCHEMA IF EXISTS $catalogwk.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/bronze", True)
dbutils.fs.rm(f"abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/silver",True)
dbutils.fs.rm(f"abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/golden",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS $catalogwk.raw;
CREATE SCHEMA IF NOT EXISTS $catalogwk.bronze;
CREATE SCHEMA IF NOT EXISTS $catalogwk.silver;
CREATE SCHEMA IF NOT EXISTS $catalogwk.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.bronze.ca_canal (
c_canal_transaccion integer,
d_canal string
)
USING DELTA
LOCATION "abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/bronze/ca_canal"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.bronze.trx_bancarias (
nu_transaccion integer
fecha_transaccion date,
nu_cliente integer,
monto numeric,
c_canal_transaccion integer
)
USING DELTA
PARTITIONED BY (nu_cliente)
LOCATION "abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/bronze/trx_bancarias"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.bronze.constructors (
  nu_transaccion integer,
  consecutivo integer,
  estatus_trx string
)
USING DELTA
LOCATION "abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/bronze/trx_verificacion_llamada"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.silver.llamadas_transaccion (
  nu_transaccion INTEGER,
  consecutivo INTEGER,
  estatus_trx string,
  probabilidad_llamada string,
  fecha_transaccion DATE,
  nu_cliente INTEGER,
  monto NUMERIC,
  c_canal_transaccion INTEGER,
  d_canal STRING
)
USING DELTA
LOCATION "abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/silver/llamadas_transaccion"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS $catalogwk.golden.estadistica_llamadas (
  nu_cliente integer,
  d_canal string,
  probabilidad_llamada integer,
  max_monto numeric,
  min_monto numeric
)
USING DELTA
LOCATION "abfss://${storageName}@lilaenriqueguadarramaosorio@gmail.com.dfs.core.windows.net/golden/llamadas_cliente_canal"